# Week 61, Day 3: Non-negative matrix factorisation: parts that add up

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w061/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: NMF by multiplicative updates

Complete nmf(X, k, g, iters), alternating the H and W multiplicative updates and recording the squared error. Factorise a 20 by 20 pixel movie of 300 frames containing four cells, and match each true footprint to its best component.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
gn = np.random.default_rng(613)
yy, xx = np.mgrid[0:20, 0:20]
centres = [(5, 6), (6, 12), (13, 7), (12, 13)]
foot = np.array([np.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * 2.5 ** 2)).ravel() for cy, cx in centres])
spk = (gn.random((300, 4)) < 0.04) * gn.uniform(1, 3, (300, 4))
common = (gn.random(300) < 0.03) * gn.uniform(1, 3, 300)
spk[:, :3] += common[:, None]                      # three cells sometimes fire together
act = np.array([np.convolve(spk[:, i], np.exp(-np.arange(40) / 8))[:300] for i in range(4)]).T
movie = act @ foot + 0.05 * np.abs(gn.normal(size=(300, 400)))

def nmf(X, k, g, iters=400):
    W = g.uniform(0.1, 1, (X.shape[0], k)); H = g.uniform(0.1, 1, (k, X.shape[1]))
    err = []
    for _ in range(iters):
        H *= ...
        W *= ...
        err.append(np.sum((X - W @ H) ** 2))
    return W, H, err

def best_match(true_rows, comps):
    C = np.corrcoef(np.vstack([true_rows, comps]))[:len(true_rows), len(true_rows):]
    return C.max(1)

W_nmf, H_nmf, nmf_err = nmf(movie, 4, gn)
match_nmf = best_match(foot, H_nmf)
print(f"relative error {nmf_err[-1] / np.sum(movie ** 2):.4f}; footprint matches {np.round(match_nmf, 3)}")
fig, ax = plt.subplots(2, 4, figsize=(8, 4))
for i in range(4):
    ax[0, i].imshow(foot[i].reshape(20, 20)); ax[1, i].imshow(H_nmf[i].reshape(20, 20))
    ax[0, i].axis("off"); ax[1, i].axis("off")
ax[0, 0].set_title("true", loc="left"); ax[1, 0].set_title("NMF", loc="left"); plt.show()

In [ ]:
import numpy as np
def test_monotone():
    "the reconstruction error never increases"
    assert all(b <= a * (1 + 1e-9) for a, b in zip(nmf_err, nmf_err[1:]))
def test_nonneg():
    "both factors are non-negative"
    assert (W_nmf >= 0).all() and (H_nmf >= 0).all()
def test_cells():
    "every true footprint is matched by an NMF component with correlation above 0.9"
    assert match_nmf.min() > 0.9

test_monotone()
test_nonneg()
test_cells()
print("All checks passed")

### Exercise: The same movie with PCA

Take the top four principal components of the centred movie as spatial maps, match them to the true footprints (allowing either sign), and measure the fraction of the components' total absolute weight that is negative (after flipping each to make its largest entry positive).

In [ ]:
Mc = movie - movie.mean(0)
_, _, Vt = np.linalg.svd(Mc, full_matrices=False)
pcs = Vt[:4]
pcs = pcs * np.sign(pcs[np.arange(4), np.abs(pcs).argmax(1)])[:, None]
match_pca = np.abs(np.corrcoef(np.vstack([foot, pcs]))[:4, 4:]).max(1)
neg_frac = ...
print(f"PCA footprint matches {np.round(match_pca, 3)} (NMF {np.round(match_nmf, 3)}); negative weight {100 * neg_frac:.0f}%")

In [ ]:
import numpy as np
def test_pca():
    "PCA components match single footprints less well than NMF components, and contain substantial negative weight"
    assert match_pca.mean() < match_nmf.mean() - 0.05 and neg_frac > 0.2

test_pca()
print("All checks passed")

### Exercise: An extra component for the neuropil

Add a diffuse background (neuropil), brighter on one side, with its own slow fluctuation. Fit NMF with 4 and with 5 components, and for each true cell find the best correlation between its activity and a column of W (using the component whose footprint matches it best).

In [ ]:
bg_fp = 0.5 + 0.5 * xx.ravel() / 19                # neuropil, brighter towards one side
bg_act = np.maximum(2 + np.convolve(np.random.default_rng(6131).normal(size=300), np.ones(30) / 30, mode="same") * 3, 0.1)
movie_bg = movie + np.outer(bg_act, bg_fp)
act_match = {}
for k in (4, 5):
    Wk, Hk, _ = nmf(movie_bg, k, np.random.default_rng(1), iters=500)
    which = np.corrcoef(np.vstack([foot, Hk]))[:4, 4:].argmax(1)
    act_match[k] = ...
    print(f"{k} components: activity correlations {np.round(act_match[k], 3)}")

In [ ]:
import numpy as np
def test_bg():
    "with a fifth component, the cells' recovered activity matches the truth clearly better than with four"
    assert np.mean(act_match[5]) > np.mean(act_match[4]) + 0.1

test_bg()
print("All checks passed")